# Lab Sains Data P8–P9 — Kelompok 2
**Abrar Mohammad Fathur (12010223140022) · Muhammad Dezra Aria (12010223140038) · Samuel Pinondang Simanjuntak (12010223140072)**

Produk Lab Pemrograman: **toko pakaian preloved/thrift**.
Keputusan: apakah ringkasan sentimen ulasan pelanggan di aplikasi toko dilabel **otomatis oleh AI (A)**, **AI + cek manusia (B)**, atau **tetap dibaca manual (C)**.
Angka pembeda: **akurasi label AI** dan **recall kelas negatif** (keluhan yang tertangkap).

Notebook ini bisa dijalankan dengan **Runtime → Run all**. Tidak perlu unggah berkas: data diambil langsung dari GitHub IndoNLP/nusax, dan label AI sudah tertanam di sel 6.

In [ ]:
# Sel 1 — pustaka dan konfigurasi
import io, re, hashlib, urllib.request
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

TANGGAL = "2026-11-06"           # tanggal ambil data (README_data)
SEED_SAMPEL, N_SAMPEL = 42, 300  # baris ini menentukan berapa banyak teks yang diambil untuk P9
SEED_CEK, N_CEK = 9, 50          # 50 label yang dicek manual oleh manusia
URL = "https://raw.githubusercontent.com/IndoNLP/nusax/main/datasets/sentiment/indonesian/{}.csv"
SPLIT_HARAP = {"train": 500, "valid": 100, "test": 400}   # jumlah baris resmi per split

## 2. Ambil data mentah dan cocokkan jumlah baris

In [ ]:
def baca(split):
    try:
        with urllib.request.urlopen(URL.format(split), timeout=30) as r:
            return pd.read_csv(io.BytesIO(r.read()))
    except Exception as e:                      # cadangan: file lokal di folder yang sama
        print(f"[{split}] gagal unduh ({e.__class__.__name__}), pakai file lokal {split}.csv")
        return pd.read_csv(f"{split}.csv")

bagian = []
for sp, n in SPLIT_HARAP.items():
    d = baca(sp)
    print(f"{sp:5s}: {len(d)} baris (harapan {n})")
    assert len(d) == n, "jumlah baris tidak cocok dengan sumber!"
    bagian.append(d.assign(split=sp))
raw = pd.concat(bagian, ignore_index=True)[["split", "id", "text", "label"]]
raw.to_csv(f"raw_{TANGGAL}.csv", index=False)     # file mentah TIDAK diubah setelah ini
print("Total:", raw.shape, "| id unik:", raw.id.is_unique)

## 3. Ringkasan data (langkah harness 15 menit)

In [ ]:
print("Baris, kolom :", raw.shape)
print("Kolom        :", list(raw.columns))
print("Nilai kosong :", raw.isna().sum().to_dict())
print("Teks duplikat:", raw.text.duplicated().sum())
print("Label        :", raw.label.value_counts().to_dict())
raw.head()

## 4. Anonimisasi (lampu kuning → hijau)
Ulasan publik masih memuat nama orang (pelayan, petugas CS, tokoh). Nama diganti `[NAMA]` sebelum teks dikirim ke AI mana pun.
Daftar nama disusun dari pemindaian pola sapaan (pak/bu/mas/mbak/kak/bang + kata), kata berhuruf kapital, dan tanda tangan CS ("Terima kasih <nama>"), lalu diperiksa manual. Nama usaha (mis. "Sate kambing H. Ihsan", "Bubur Ahong") dan merek dibiarkan.

In [ ]:
NAMA = ['dela','isna','diah','gina','tisna','echa','tian','luqman','elsa','adit','doni','rocky',
        'farrel','dian','nadia','lulung','rohayani','arif']
pola = re.compile(r'\b(' + '|'.join(NAMA) + r')\b', re.I)
ttd  = re.compile(r'([Tt]erima kasih,?\s+)(zan|era)(\s*\.?\s*)$')   # tanda tangan petugas CS

bersih = raw.copy()
bersih["n_nama_diganti"] = bersih.text.str.count(pola) + bersih.text.str.count(ttd)
bersih["text"] = (bersih.text.str.replace(pola, "[NAMA]", regex=True)
                              .str.replace(ttd, lambda m: m.group(1) + "[NAMA]" + m.group(3), regex=True))
bersih.to_csv(f"bersih_{TANGGAL}.csv", index=False)
print("Baris yang diubah:", (bersih.n_nama_diganti > 0).sum(), "| total nama diganti:", bersih.n_nama_diganti.sum())
print("Cek sisa pola kontak (@, URL, no HP, email):",
      bersih.text.str.contains(r'@\w+|http|www\.|(?:\+62|08)\d{7,}', regex=True).sum())

## 5. Sampel 300 teks untuk dilabel AI (tanpa label asli)

In [ ]:
sampel = bersih.sample(n=N_SAMPEL, random_state=SEED_SAMPEL).reset_index(drop=True)
sampel.insert(0, "no", range(1, N_SAMPEL + 1))
sampel[["no", "text"]].to_csv("teks_untuk_AI.csv", index=False)   # hanya ini yang dikirim ke AI
print(sampel.label.value_counts().to_dict(), sampel.split.value_counts().to_dict())

## 6. Label dari AI
AI: Claude (Anthropic), sesi 6 Nov 2026. AI hanya menerima `no` + `text` dari `teks_untuk_AI.csv`, tanpa label asli.
Prompt: *"Beri satu label sentimen untuk tiap teks: positive, neutral, atau negative. Sentimen = sikap penulis. Promosi/iklan/fakta tanpa sikap = neutral. Ulasan campuran = sikap yang dominan."*
Format di bawah: `<id NusaX><kode>`; P = positive, N = neutral, G = negative.

In [ ]:
LABEL_AI = """
30P 3P 617P 745N 387P 59N 548P 50P 640P 297N 735G 935P 971G 91P 353G 747G 845G 174P 349G
176N 188N 929G 332G 787G 379P 381G 339G 367N 497N 585G 75G 257N 62G 719G 609G 715P 731P
322G 795P 537P 58G 748P 952P 611G 788G 274N 164G 729N 428G 423G 960N 5P 752P 192P 707G
852P 462G 13P 11P 390G 692N 621N 344P 581N 177N 80N 163N 134P 158G 97N 210N 161G 784P
15G 837P 684G 294G 216N 782G 643N 742P 18N 197P 622N 471N 608N 407G 195G 70G 650N 775N
4G 389G 857P 485G 671P 151P 571P 124P 998N 443P 526G 361G 808G 994G 175N 533N 658P 382P
439N 515P 400P 160G 347G 557P 681N 229G 321P 931G 577N 49G 199G 910P 153N 418P 455P 308G
882G 738P 806P 873N 598P 896G 502P 385G 820P 231P 414G 855G 318G 374G 689N 915G 905G
987G 350G 463P 334P 121G 934N 947G 187N 932P 809P 644G 783G 217P 21P 733P 313G 335N 575N
288P 667G 753G 711G 755P 714G 912P 901G 722N 736G 279N 885G 362G 35G 777N 544N 249N 383P
899G 580G 565P 110N 221N 277P 133P 25G 962G 241G 194G 928P 86G 641G 261G 542G 450G 672G
477N 572P 467P 720N 538G 730P 831G 921P 574N 991G 589G 771P 793P 72N 459P 770P 633G 470P
646G 139G 162G 244G 0G 813P 535P 676P 835G 436N 964P 880N 688P 409P 7P 148G 763G 102P
181P 861N 612G 859P 807P 930N 394P 83N 868N 509G 647G 398G 826G 607G 404N 284G 290P 406G
913P 472P 907G 245G 482G 686G 599G 317N 638N 17G 494G 90P 546P 430P 970G 71N 146N 425G
413N 397P 997N 84N 495G 504P 172P 262P 977P 315G 125P 884G 78P 554G 858P 682P 798N 433G
805P 324P 320P 205N 101G 520G 232N 45G 988P 893P 751G 506N
"""
kode = {"P": "positive", "N": "neutral", "G": "negative"}
ai = {int(t[:-1]): kode[t[-1]] for t in LABEL_AI.split()}
assert len(ai) == N_SAMPEL and set(ai) == set(sampel.id), "id label AI tidak cocok dengan sampel"
sampel["label_ai"] = sampel.id.map(ai)
sampel["benar"] = sampel.label == sampel.label_ai
sampel.to_csv("label_ai_300.csv", index=False)
sampel.label_ai.value_counts()

## 7. Seberapa sering AI benar? (dibanding label pakar NusaX, 300 teks)

In [ ]:
L = ["negative", "neutral", "positive"]
cm = pd.crosstab(sampel.label, sampel.label_ai).reindex(index=L, columns=L, fill_value=0)
cm.index.name, cm.columns.name = "label pakar", "label AI"
akurasi = sampel.benar.mean()
rng = np.random.default_rng(0)
ci = np.percentile([rng.choice(sampel.benar.values, N_SAMPEL).mean() for _ in range(5000)], [2.5, 97.5])

def kappa(a, b):
    po = (a == b).mean()
    pe = sum((a == k).mean() * (b == k).mean() for k in L)
    return (po - pe) / (1 - pe)

ringkas = pd.DataFrame({
    "recall":    [cm.loc[k, k] / cm.loc[k].sum() for k in L],
    "precision": [cm.loc[k, k] / cm[k].sum() for k in L],
    "n_pakar":   [cm.loc[k].sum() for k in L]}, index=L).round(3)
print(f"Akurasi: {sampel.benar.sum()}/{N_SAMPEL} = {akurasi:.1%}  (IK 95% bootstrap {ci[0]:.1%}–{ci[1]:.1%})")
print(f"Cohen's kappa: {kappa(sampel.label, sampel.label_ai):.3f}")
display(cm); display(ringkas)

fig, ax = plt.subplots(figsize=(4.2, 3.6))
ax.imshow(cm.values, cmap="Blues")
for i in range(3):
    for j in range(3):
        ax.text(j, i, cm.values[i, j], ha="center", va="center",
                color="white" if cm.values[i, j] > 60 else "black")
ax.set_xticks(range(3), L); ax.set_yticks(range(3), L)
ax.set_xlabel("label AI"); ax.set_ylabel("label pakar NusaX"); ax.set_title("Matriks kebingungan, n = 300")
plt.tight_layout(); plt.show()

In [ ]:
# Daftar label AI yang tidak sama dengan label pakar (bahan diskusi lisan)
sampel.loc[~sampel.benar, ["no", "id", "label", "label_ai", "text"]]

## 8. Lembar cek 50 label oleh manusia (tugas P9)
Sel ini membuat `cek50_manusia.csv`. **Isi kolom `label_manusia` sendiri tanpa melihat label pakar**, simpan sebagai `cek50_terisi.csv`, unggah ke Colab, lalu jalankan ulang sel ini.

In [ ]:
cek = sampel.sample(n=N_CEK, random_state=SEED_CEK).sort_values("no")
cek[["no", "id", "text", "label_ai"]].assign(label_manusia="", catatan="").to_csv("cek50_manusia.csv", index=False)
print(f"Pembanding pakar pada 50 baris yang sama: {cek.benar.sum()}/{N_CEK} = {cek.benar.mean():.0%}")

import os
if os.path.exists("cek50_terisi.csv"):
    h = pd.read_csv("cek50_terisi.csv").dropna(subset=["label_manusia"])
    h["label_manusia"] = h.label_manusia.str.strip().str.lower()
    setuju = (h.label_manusia == h.label_ai)
    print(f"AI benar menurut manusia: {setuju.sum()}/{len(h)} = {setuju.mean():.0%}")
    print(f"Manusia vs pakar sepakat: {(h.merge(cek[['no','label']], on='no').eval('label_manusia == label')).mean():.0%}")
else:
    print("cek50_terisi.csv belum ada — isi lembar cek dulu.")

## 9. Terapkan ambang keputusan (ditetapkan di Lembar Rencana Data sebelum pelabelan)

In [ ]:
rec_neg = ringkas.loc["negative", "recall"]
if akurasi >= 0.85 and rec_neg >= 0.80:
    pilihan = "A — label otomatis AI (dengan audit sampel berkala)"
elif akurasi >= 0.70:
    pilihan = "B — AI melabel, manusia cek label negatif/netral"
else:
    pilihan = "C — tetap dibaca manual"
print(f"Akurasi {akurasi:.1%}, recall negatif {rec_neg:.1%} → {pilihan}")
print("Catatan: ini data proksi (NusaX, bukan ulasan toko thrift). Keputusan final menunggu cek 50 manusia + uji ulang di ulasan toko sendiri.")